# Final Assignment: Part 2 - Create Dashboard with Plotly and Dash

This notebook builds the **Automobile Sales Statistics Dashboard** using Plotly and Dash, covering:

- **4.1** — A Dash application with a meaningful title
- **4.2** — Drop-down menus with appropriate titles and options
- **4.3** — A division (`html.Div`) for output display with an appropriate `id` and `className`
- **4.4** — A callback function that updates the output container based on the selected statistics

**Note on running in Colab:** Dash apps normally run their own local web server (`app.run()`), which doesn't work the same way inside a hosted Colab kernel. Two cells below are provided:
1. The **exact script** you should save as a `.py` file and upload for grading (this matches what the assignment expects, e.g. running via `python3.11 automobile_dashboard.py` in the IBM Skills Network lab environment).
2. An optional **Colab-only launcher cell** using `jupyter-dash` so you can preview the dashboard directly inside this notebook.


## 1. Install/Import required libraries

In [ ]:
# Run this once in Colab to make sure the libraries are available
!pip install dash pandas plotly -q


## 2. The Python script (save/download this as `automobile_dashboard.py` for submission)

Everything in the cell below is the complete script. It satisfies all four rubric items:

- `app = dash.Dash(__name__)` + `app.title` + an `html.H1(...)` heading → **4.1 meaningful title**
- Two `dcc.Dropdown` components with `Label`s, `options`, and `placeholder` → **4.2 dropdown menus**
- `html.Div(id='output-container', className='chart-grid', ...)` → **4.3 output division with id + className**
- `@app.callback` on `output-container` driven by the dropdown selections → **4.4 callback function**


In [ ]:
%%writefile automobile_dashboard.py
# ---------------------------------------------------------------
# Final Assignment Part 2: Automobile Sales Statistics Dashboard
# Built with Plotly and Dash
# ---------------------------------------------------------------

# Import required libraries
import pandas as pd
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# -----------------------------------------------------------------------------
# Load the data using pandas
# -----------------------------------------------------------------------------
data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
    'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/'
    'historical_automobile_sales.csv'
)

# -----------------------------------------------------------------------------
# Initialize the Dash app
# -----------------------------------------------------------------------------
app = dash.Dash(__name__)

# TASK 4.1: Give the dashboard a meaningful title
app.title = "Automobile Sales Statistics Dashboard"

# -----------------------------------------------------------------------------
# Dropdown option lists
# -----------------------------------------------------------------------------
dropdown_options = [
    {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
    {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
]

year_list = [i for i in range(1980, 2024, 1)]

# -----------------------------------------------------------------------------
# TASK 4.1 / 4.2 / 4.3: App layout - title, dropdowns, and output division
# -----------------------------------------------------------------------------
app.layout = html.Div([

    # TASK 4.1: Meaningful title displayed at the top of the dashboard
    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={'textAlign': 'center', 'color': '#503D36', 'font-size': 24}
    ),

    # TASK 4.2: Dropdown menu #1 - choose the type of report/statistics
    html.Div([
        html.Label("Select Statistics:"),
        dcc.Dropdown(
            id='dropdown-statistics',
            options=dropdown_options,
            value='Yearly Statistics',
            placeholder='Select a report type',
            style={'width': '80%', 'padding': '3px', 'font-size': '20px',
                   'text-align-last': 'center'}
        )
    ]),

    # TASK 4.2: Dropdown menu #2 - choose the year
    html.Div(
        dcc.Dropdown(
            id='select-year',
            options=[{'label': i, 'value': i} for i in year_list],
            value=year_list[0],
            placeholder='Select Year',
            style={'width': '80%', 'padding': '3px', 'font-size': '20px',
                   'text-align-last': 'center'}
        )
    ),

    # TASK 4.3: Division for output display - appropriate id and className
    html.Div([
        html.Div(
            id='output-container',
            className='chart-grid',
            style={'display': 'flex', 'flex-direction': 'column'}
        )
    ])
])


# -----------------------------------------------------------------------------
# TASK 4.4: Callback - enable/disable the year dropdown based on report type
# -----------------------------------------------------------------------------
@app.callback(
    Output(component_id='select-year', component_property='disabled'),
    Input(component_id='dropdown-statistics', component_property='value')
)
def update_input_container(selected_statistics):
    if selected_statistics == 'Yearly Statistics':
        return False
    else:
        return True


# -----------------------------------------------------------------------------
# TASK 4.4: Callback - update the output container based on the selected
# statistics AND the selected year
# -----------------------------------------------------------------------------
@app.callback(
    Output(component_id='output-container', component_property='children'),
    [Input(component_id='dropdown-statistics', component_property='value'),
     Input(component_id='select-year', component_property='value')]
)
def update_output_container(selected_statistics, input_year):

    if selected_statistics == 'Recession Period Statistics':
        recession_data = data[data['Recession'] == 1]

        # Plot 1: Average automobile sales over the recession period (line)
        yearly_rec = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        R_chart1 = dcc.Graph(
            figure=px.line(
                yearly_rec, x='Year', y='Automobile_Sales',
                title="Average Automobile Sales fluctuation over Recession Period"
            )
        )

        # Plot 2: Average vehicles sold by vehicle type during recession (bar)
        average_sales = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        R_chart2 = dcc.Graph(
            figure=px.bar(
                average_sales, x='Vehicle_Type', y='Automobile_Sales',
                title="Average Vehicles Sold by Vehicle Type during Recession"
            )
        )

        # Plot 3: Advertising expenditure share by vehicle type (pie)
        exp_rec = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        R_chart3 = dcc.Graph(
            figure=px.pie(
                exp_rec, values='Advertising_Expenditure', names='Vehicle_Type',
                title="Total Advertisement Expenditure Share by Vehicle Type during Recession"
            )
        )

        # Plot 4: Effect of unemployment rate on vehicle type and sales (bar)
        unemp_data = recession_data.groupby(['unemployment_rate', 'Vehicle_Type'])['Automobile_Sales'].mean().reset_index()
        R_chart4 = dcc.Graph(
            figure=px.bar(
                unemp_data, x='unemployment_rate', y='Automobile_Sales',
                color='Vehicle_Type',
                labels={'unemployment_rate': 'Unemployment Rate',
                        'Automobile_Sales': 'Average Automobile Sales'},
                title='Effect of Unemployment Rate on Vehicle Type and Sales'
            )
        )

        return [
            html.Div(className='chart-item',
                     children=[html.Div(children=R_chart1), html.Div(children=R_chart2)],
                     style={'display': 'flex'}),
            html.Div(className='chart-item',
                     children=[html.Div(children=R_chart3), html.Div(children=R_chart4)],
                     style={'display': 'flex'})
        ]

    elif input_year and selected_statistics == 'Yearly Statistics':
        yearly_data = data[data['Year'] == input_year]

        # Plot 1: Yearly automobile sales for the whole period (line)
        yas = data.groupby('Year')['Automobile_Sales'].mean().reset_index()
        Y_chart1 = dcc.Graph(
            figure=px.line(yas, x='Year', y='Automobile_Sales',
                            title='Yearly Average Automobile Sales')
        )

        # Plot 2: Total monthly automobile sales for the selected year (line)
        mas = yearly_data.groupby('Month')['Automobile_Sales'].sum().reset_index()
        Y_chart2 = dcc.Graph(
            figure=px.line(mas, x='Month', y='Automobile_Sales',
                            title='Total Monthly Automobile Sales')
        )

        # Plot 3: Average vehicles sold by vehicle type in selected year (bar)
        avr_vdata = yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
        Y_chart3 = dcc.Graph(
            figure=px.bar(
                avr_vdata, x='Vehicle_Type', y='Automobile_Sales',
                title='Average Vehicles Sold by Vehicle Type in {}'.format(input_year)
            )
        )

        # Plot 4: Advertisement expenditure by vehicle type in selected year (pie)
        exp_data = yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
        Y_chart4 = dcc.Graph(
            figure=px.pie(
                exp_data, values='Advertising_Expenditure', names='Vehicle_Type',
                title='Total Advertisement Expenditure by Vehicle Type in {}'.format(input_year)
            )
        )

        return [
            html.Div(className='chart-item',
                     children=[html.Div(children=Y_chart1), html.Div(children=Y_chart2)],
                     style={'display': 'flex'}),
            html.Div(className='chart-item',
                     children=[html.Div(children=Y_chart3), html.Div(children=Y_chart4)],
                     style={'display': 'flex'})
        ]

    else:
        return None


# -----------------------------------------------------------------------------
# Run the app (use this form when running the .py file directly, e.g. in the
# IBM Skills Network lab terminal: python3.11 automobile_dashboard.py)
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    app.run(debug=True)


## 3. (Optional) Preview the dashboard inside Colab

Dash's normal `app.run()` opens a local server on `127.0.0.1`, which Colab's hosted runtime can't open a browser tab to directly. To preview *inside this notebook*, install `jupyter-dash` and use `app.run(mode='inline')` — this is only for previewing; it is **not** part of the graded script.


In [ ]:
# Optional: preview only (not required for the submitted .py file)
!pip install jupyter-dash -q

import runpy
# Re-run the script's definitions in this notebook's namespace so we can
# override how the app launches, without touching automobile_dashboard.py
namespace = runpy.run_path('automobile_dashboard.py', run_name='not_main')
app = namespace['app']

# Launch inline inside the notebook (Dash 2.11+ supports app.run(jupyter_mode='inline'))
app.run(jupyter_mode='inline', port=8050)


## 4. Download the script for submission

Run the cell below, then use the Colab file browser (folder icon on the left) to download **`automobile_dashboard.py`** — that is the file to upload for Question 4.


In [ ]:
from google.colab import files
files.download('automobile_dashboard.py')
